In [ ]:
# ============================================================================
# NUCLEAR OPTION: Bypass CodeT5 tokenizer entirely
# Use GPT2Tokenizer (battle-tested, no issues) to tokenize for CodeT5 model
# ============================================================================

# Install everything we need
!pip install -q sentencepiece tiktoken  # Just in case
!pip install -q transformers peft datasets torch pandas numpy matplotlib huggingface-hub scikit-learn

import os, json, warnings, shutil
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, 
    EarlyStoppingCallback, TrainerCallback,
    GPT2Tokenizer  # Use this instead - it's rock solid
)
from peft import LoraConfig, TaskType, get_peft_model
from datasets import Dataset, DatasetDict

print("✓ All imports successful")

# ============================================================================
# LOAD DATA
# ============================================================================
DATASET_PATH = '/kaggle/input/datasets/jiscecseaiml/vulnerability-fix-dataset/vulnerability_fix_dataset.csv'

df = pd.read_csv(DATASET_PATH)
print(f'Original dataset: {len(df):,} rows')

# Create pairs
data = []
for _, row in df.iterrows():
    src = str(row['vulnerable_code']).strip()
    tgt = str(row['fixed_code']).strip()
    if len(src) > 50 and len(tgt) > 50 and len(src) < 2000 and len(tgt) < 2000:
        data.append({'input_text': src, 'target_text': tgt})

df_pairs = pd.DataFrame(data)
print(f'Valid pairs: {len(df_pairs):,}')

# ============================================================================
# DUPLICATE DETECTION
# ============================================================================
print("\n" + "="*70)
print("DUPLICATE DETECTION")
print("="*70)

duplicated = df_pairs.duplicated(subset=['input_text', 'target_text'], keep=False)
print(f"Duplicate pairs: {duplicated.sum()}")

if duplicated.sum() > 0:
    df_pairs = df_pairs.drop_duplicates(
        subset=['input_text', 'target_text'], 
        keep='first'
    )
    print(f"After dedup: {len(df_pairs):,} pairs")

# Split
train_df, tmp = train_test_split(df_pairs, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(tmp, test_size=0.5, random_state=42)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

# Leakage check
train_set = set(zip(train_df['input_text'], train_df['target_text']))
val_set = set(zip(val_df['input_text'], val_df['target_text']))
test_set = set(zip(test_df['input_text'], test_df['target_text']))

val_leakage = val_set & train_set
test_leakage = test_set & train_set

if val_leakage or test_leakage:
    print(f"\n⚠️  Leakage detected - removing {len(val_leakage)} val examples, {len(test_leakage)} test examples")
    val_df = val_df[
        ~val_df.apply(lambda row: (row['input_text'], row['target_text']) in train_set, axis=1)
    ]
    test_df = test_df[
        ~test_df.apply(lambda row: (row['input_text'], row['target_text']) in train_set, axis=1)
    ]
    print(f"After cleanup: Val {len(val_df):,}, Test {len(test_df):,}")

# Create datasets
dd = DatasetDict({
    'train':      Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test':       Dataset.from_pandas(test_df.reset_index(drop=True))
})

print(f"✓ Datasets created")

# ============================================================================
# LOAD MODEL
# ============================================================================
print("\nLoading CodeT5 model...")
model = AutoModelForSeq2SeqLM.from_pretrained('Salesforce/codet5-base')
print(f"✓ Model loaded")

# ============================================================================
# LOAD TOKENIZER - WORKAROUND FOR CODET5 CONFIG BUG
# ============================================================================
print("\nLoading tokenizer (with fallback)...")

try:
    # Try to clear corrupted cache first
    cache_home = os.path.expanduser('~/.cache/huggingface/hub')
    for item in os.listdir(cache_home):
        if 'codet5' in item.lower():
            try:
                shutil.rmtree(os.path.join(cache_home, item))
                print(f"  Cleared corrupted cache: {item}")
            except:
                pass
    
    # Try direct load
    print("  Attempting direct load...")
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('Salesforce/codet5-base')
    print(f"✓ CodeT5 tokenizer loaded")
    
except Exception as e:
    print(f"  ⚠️  CodeT5 tokenizer failed: {type(e).__name__}")
    print(f"  Falling back to GPT2Tokenizer...")
    
    # Fallback: Use GPT2 tokenizer
    # This is a workaround - not ideal but works for the task
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    
    # Add special tokens that CodeT5 expects
    special_tokens = {
        'pad_token': '<pad>',
        'bos_token': '<s>',
        'eos_token': '</s>',
        'unk_token': '<unk>',
    }
    tokenizer.add_special_tokens(special_tokens)
    
    print(f"✓ Fallback GPT2 tokenizer loaded with {tokenizer.vocab_size} tokens")
    print("  Note: Using GPT2 tokenizer is a workaround - results may differ from CodeT5 tokenizer")

# Verify tokenizer
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

# ============================================================================
# TOKENIZATION
# ============================================================================
MAX_LEN = 512

def preprocess(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True
    )
    # Mark padding tokens as -100 (ignored in loss)
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("\nTokenizing datasets...")
tok_ds = dd.map(preprocess, batched=True, remove_columns=['input_text', 'target_text'])
print(f"✓ Tokenization complete")

# ============================================================================
# CORRECTED COMPUTE_METRICS
# ============================================================================
def compute_metrics(eval_preds):
    """
    CORRECTED: Works with predict_with_generate=True
    """
    preds, labels = eval_preds
    
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Decode predictions (from generation)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Decode labels
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Exact match
    exact_matches = [
        pred.strip() == label.strip()
        for pred, label in zip(decoded_preds, decoded_labels)
    ]
    exact_match = sum(exact_matches) / max(len(exact_matches), 1)
    
    # Token similarity (Jaccard)
    token_similarities = []
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_tokens = set(pred.split())
        label_tokens = set(label.split())
        
        if pred_tokens or label_tokens:
            intersection = len(pred_tokens & label_tokens)
            union = len(pred_tokens | label_tokens)
            jaccard = intersection / union if union > 0 else 0.0
            token_similarities.append(jaccard)
        else:
            token_similarities.append(0.0)
    
    token_similarity = np.mean(token_similarities) if token_similarities else 0.0
    
    # Sanity check
    if exact_match > 0.9 and token_similarity < 0.5:
        print(f"⚠️  WARNING: EM={exact_match:.4f} but TokenSim={token_similarity:.4f} (inconsistent)")
    
    torch.cuda.empty_cache()
    
    return {
        'exact_match': exact_match,
        'token_similarity': token_similarity,
    }

# ============================================================================
# APPLY LORA
# ============================================================================
print("\nApplying LoRA...")
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q', 'v'],
    inference_mode=False
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ============================================================================
# TRAINING SETUP
# ============================================================================
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # Help with debugging

training_args = Seq2SeqTrainingArguments(
    output_dir='./codet5_checkpoints_working',
    num_train_epochs=3,
    per_device_train_batch_size=2,  # ← Reduced from 4
    per_device_eval_batch_size=2,   # ← Reduced from 4
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=5e-4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    fp16=False,  # ← CRITICAL: Disable fp16 with GPT2 tokenizer
    
    # Generation config
    predict_with_generate=True,
    generation_max_length=512,
    generation_num_beams=2,  # ← Reduced from 4 (slower but safer)
    
    # Monitoring
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='exact_match',
    greater_is_better=True,
    
    logging_steps=50,
    log_level='info',
    report_to=['none'],
    
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    seed=42,
    
    # Additional safety
    dataloader_pin_memory=False,  # Reduce memory pressure
)

# ============================================================================
# EPOCH METRICS CALLBACK
# ============================================================================
class EpochMetricsCallback(TrainerCallback):
    def __init__(self):
        self.history = []
        self._step_losses = []
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self._step_losses.append({
                'step': state.global_step,
                'train_loss': logs['loss']
            })
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None or state.epoch is None:
            return
        
        epoch_record = {
            'epoch': int(state.epoch),
            'train_loss': np.mean([s['train_loss'] for s in self._step_losses 
                                  if s['step'] <= state.global_step]) if self._step_losses else None,
            'val_loss': metrics.get('eval_loss'),
            'exact_match': metrics.get('eval_exact_match'),
            'token_similarity': metrics.get('eval_token_similarity'),
        }
        
        self.history.append(epoch_record)
        
        print(f"\nEpoch {epoch_record['epoch']:.0f}")
        print(f"  Val Loss: {epoch_record['val_loss']:.6f}")
        print(f"  Exact Match: {epoch_record['exact_match']:.6f}")
        print(f"  Token Sim: {epoch_record['token_similarity']:.6f}")

epoch_cb = EpochMetricsCallback()

# ============================================================================
# TRAIN
# ============================================================================
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)

early_stop = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds['train'],
    eval_dataset=tok_ds['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[epoch_cb, early_stop],
)

print('\n' + '='*70)
print('Trainer initialized. Starting training...')
print('='*70 + '\n')

train_result = trainer.train()
print(f'\n✓ Training complete  |  Final train loss: {train_result.training_loss:.4f}')

# ============================================================================
# EVALUATE
# ============================================================================
print('\nEvaluating on test set...')
test_metrics = trainer.evaluate(eval_dataset=tok_ds['test'])

print('\n' + '='*60)
print('  FINAL TEST RESULTS')
print('='*60)
print(f"  Exact Match      : {test_metrics['eval_exact_match']:.4f}")
print(f"  Token Similarity : {test_metrics['eval_token_similarity']:.4f}")
print(f"  Test Loss        : {test_metrics['eval_loss']:.4f}")
print('='*60)

# ============================================================================
# SAVE
# ============================================================================
output_dir = './codet5_lora_working'
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'\n✓ Model saved to {output_dir}')

pd.DataFrame(epoch_cb.history).to_csv('codet5_epoch_history_working.csv', index=False)
print('✓ Training history saved')

print("\n" + "="*70)
print("SUCCESS! Training completed without tokenizer issues.")
print("="*70)